# Perplexity

In [ ]:
import torch
from transformers import (
    GemmaConfig,
    AutoModelForCausalLM,
    GemmaTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    default_data_collator,
)
from datasets import load_dataset
import math

# CONFIG
PRETRAINED_CHECKPOINT_FILE = "./data/gemma20m_ckpts/ckpt-83000.pt"
TOKENIZER_FILE = "spm_multilang.model"
# Use your original tokenized validation set for this
EVAL_DATA_FILE = "./data/multilang_val.tok"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# LOAD MODEL AND TOKENIZER
print("Loading Pre-trained Model and Tokenizer")
ckpt = torch.load(PRETRAINED_CHECKPOINT_FILE, map_location="cpu")
config = GemmaConfig(**ckpt["config"])
model = AutoModelForCausalLM.from_config(config)
model.load_state_dict(ckpt["model"], strict=False)
tokenizer = GemmaTokenizer(vocab_file=TOKENIZER_FILE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.to(DEVICE)

# LOAD AND PREPARE DATA
# We need to re-process the .tok file into the right format
raw_dataset = load_dataset("text", data_files={"validation": EVAL_DATA_FILE})
def tokenize_function(examples):
    # The data is already token IDs, so we just convert to lists of ints
    token_lists = [[int(t) for t in line.split()] for line in examples["text"]]
    # The collator will handle padding and creating labels
    return {"input_ids": token_lists}

eval_dataset = raw_dataset['validation'].map(tokenize_function, batched=True, remove_columns=["text"])
# Use a small subset for a fast evaluation
eval_dataset = eval_dataset.select(range(2000))

# EVALUATE
trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="./eval_temp", per_device_eval_batch_size=8),
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False), # <-- Change this
)

print("\nCalculating Perplexity")
metrics = trainer.evaluate()
perplexity = math.exp(metrics["eval_loss"])

print(f"\nEvaluation Complete")
print(f"Validation Loss: {metrics['eval_loss']:.4f}")
print(f"Perplexity: {perplexity:.4f}")

--- Loading Pre-trained Model and Tokenizer ---


/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_75222/4158740151.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



--- Calculating Perplexity ---



--- Evaluation Complete ---
Validation Loss: 7.9576
Perplexity: 2857.0811


In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# --- CONFIG ---
FINETUNED_MODEL_DIR = "./final_gemma270m_finetuned" # Path to your final fine-tuned model
VALID_DATA_FILE = "./data/data_multilingual/valid.jsonl"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# --- LOAD MODEL AND TOKENIZER ---
print(f"--- Loading Fine-tuned Model from {FINETUNED_MODEL_DIR} ---")
tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(FINETUNED_MODEL_DIR)
model.to(DEVICE)
model.eval()

# --- LOAD VALIDATION DATA ---
valid_dataset = load_dataset("json", data_files=VALID_DATA_FILE, split="train")
# Use a smaller subset for a fast evaluation
valid_dataset = valid_dataset.select(range(500))

# --- CALCULATE ACCURACY ---
correct_predictions = 0
total_predictions = 0

print("\n--- Calculating Exact Match Accuracy ---")
for example in tqdm(valid_dataset):
    # Get the ground truth answer
    ground_truth_answer = example["messages"][1]["content"]

    # Create the prompt
    prompt = f"user: {example['messages'][0]['content']}\nassistant: "
    
    # Generate a response
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20, eos_token_id=tokenizer.eos_token_id)
    
    # Decode the generated text
    prediction = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # Check for an exact match
    if prediction == ground_truth_answer:
        correct_predictions += 1
    total_predictions += 1

accuracy = (correct_predictions / total_predictions) * 100

print(f"\n--- Evaluation Complete ---")
print(f"Correct Predictions: {correct_predictions} / {total_predictions}")
print(f"Exact Match Accuracy: {accuracy:.2f}%")

--- Loading Fine-tuned Model from ./final_gemma270m_finetuned ---

--- Calculating Exact Match Accuracy ---


100%|██████████| 500/500 [07:40<00:00,  1.09it/s]


--- Evaluation Complete ---
Correct Predictions: 0 / 500
Exact Match Accuracy: 0.00%
